# Chapter 33: Training, Validation, Testing, and Cross-validation

This notebook builds reproducible evaluation splits for NRG shipment records.


In [1]:
import matplotlib.pyplot as plt
from datasciencebook.evaluation_splits import (
    holdout_indices, kfold_indices, grouped_holdout,
    expanding_window_splits, summarize_scores, select_lowest_loss,
)


## 1. Create a reproducible holdout


In [2]:
split = holdout_indices(10, train_fraction=0.6, validation_fraction=0.2, seed=7)
print(split)
print('All rows used once:', sorted(split['train'] + split['validation'] + split['test']) == list(range(10)))


{'train': [8, 3, 1, 4, 7, 0], 'validation': [9, 6], 'test': [2, 5]}
All rows used once: True


## 2. Inspect five-fold cross-validation


In [3]:
folds = kfold_indices(10, folds=5, shuffle=False)
for number, (train, valid) in enumerate(folds, start=1):
    print(number, 'train', train, 'validate', valid)


1 train [1, 6, 2, 7, 3, 8, 4, 9] validate [0, 5]
2 train [0, 5, 2, 7, 3, 8, 4, 9] validate [1, 6]
3 train [0, 5, 1, 6, 3, 8, 4, 9] validate [2, 7]
4 train [0, 5, 1, 6, 2, 7, 4, 9] validate [3, 8]
5 train [0, 5, 1, 6, 2, 7, 3, 8] validate [4, 9]


## 3. Keep order rows together


In [4]:
groups = ['O1', 'O1', 'O2', 'O3', 'O3', 'O4']
train, test = grouped_holdout(groups, {'O3'})
print('Training indices:', train)
print('Test indices:', test)
print('Shared groups:', set(groups[i] for i in train) & set(groups[i] for i in test))


Training indices: [0, 1, 2, 5]
Test indices: [3, 4]
Shared groups: set()


## 4. Build expanding time windows with a gap


In [5]:
time_splits = expanding_window_splits(20, initial_train=8, validation_size=3, step=3, gap=1)
for number, (train, valid) in enumerate(time_splits, start=1):
    print(number, f'train 0-{train[-1]}', f'validate {valid[0]}-{valid[-1]}')


1 train 0-7 validate 9-11
2 train 0-10 validate 12-14
3 train 0-13 validate 15-17


## 5. Summarise fold losses and select a setting


In [6]:
scores = summarize_scores([0.24, 0.28, 0.22, 0.31])
choice = select_lowest_loss({'baseline': 0.29, 'small_tree': 0.25, 'large_tree': 0.27})
print('Score summary:', scores)
print('Selected setting:', choice)


Score summary: {'folds': 4, 'mean': 0.2625, 'minimum': 0.22, 'maximum': 0.31, 'range': 0.09}
Selected setting: small_tree


## 6. Visualise the temporal evaluation windows


In [7]:
fig, ax = plt.subplots()
for row, (train, valid) in enumerate(time_splits):
    ax.scatter(train, [row] * len(train), marker='s', label='Training' if row == 0 else None, color='#4472C4')
    ax.scatter(valid, [row] * len(valid), marker='s', label='Validation' if row == 0 else None, color='#C55A11')
ax.set_xlabel('Week index')
ax.set_ylabel('Split')
ax.set_yticks(range(len(time_splits)), [f'Split {i}' for i in range(1, len(time_splits) + 1)])
ax.set_title('NRG expanding-window validation')
ax.legend()
plt.tight_layout()
plt.show()


<Figure size 640x480 with 1 Axes>

## Practice

Create a grouped evaluation for a new-warehouse use case and explain why row-level random splitting would answer a different question.


In [ ]:
# Create the grouped practice split here.
